# 對抗資料產生：Tone-Neutralization via Gemini

本 notebook 從訓練集中抽取標題黨樣本（label=1），透過 Gemini API 改寫成平實、客觀的非標題黨版本，
產生對抗資料對 `(orig_title, plain_title)` 並輸出至 `dataset/processed/adversarial_tone.csv`。

## Step 1：抽樣

讀取訓練集，篩選 label==1（標題黨），依語言（zh / en）各抽 400 筆。

In [1]:
import pandas as pd

train = pd.read_csv("../dataset/processed/train.csv")
click = train[train["label"] == 1]
zh = click[click["lang"] == "zh"].sample(n=400, random_state=42)
en = click[click["lang"] == "en"].sample(n=400, random_state=42)
sampled = pd.concat([zh, en]).reset_index(drop=True)
print(sampled["lang"].value_counts())

lang
zh    400
en    400
Name: count, dtype: int64


## Step 2：Gemini 改寫

使用 `google.genai` SDK（新版）呼叫 Gemini，將標題黨標題改寫為中性、客觀的新聞標題（label=0 的對應版本）。

需要設定環境變數 `GEMINI_API_KEY`。

In [2]:
import os, time, json
from pathlib import Path
from dotenv import load_dotenv
from google import genai

load_dotenv(Path("../backend/.env"))

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL_ID = "gemini-3.1-flash-lite-preview"
BATCH_SIZE = 1
SLEEP_BETWEEN_BATCHES = 8
MAX_RETRIES = 5
RETRY_SLEEP = 20

CHECKPOINT_PATH = Path("../dataset/processed/_rewrite_checkpoint.jsonl")

def rewrite_batch(batch_rows, lang):
    numbered = "\n".join(
        f"{i+1}. 標題：{r['title']}\n   內文：{str(r.get('content', ''))[:300]}"
        for i, r in enumerate(batch_rows)
    )
    if lang == "zh":
        prompt = f"""以下有 {len(batch_rows)} 個標題，請將每個標題改寫成中性、客觀、可作為非標題黨新聞標題的一句繁體中文。
要求：忠實反映內文、不得加入內文沒有的資訊；去除誇飾詞、懸念、問句、感嘆、模糊代稱。
只輸出改寫後標題，每行一個，格式為「編號. 改寫標題」，不要任何其他說明。

{numbered}

輸出："""
    else:
        prompt = f"""Rewrite each of the following {len(batch_rows)} headlines into a neutral, objective, non-clickbait news headline in English.
Requirements: faithfully reflect the content; do not add information absent from the content; remove hype, curiosity gaps, questions, exclamations, vague pronouns.
Output only the rewritten headlines, one per line, in format "N. rewritten headline", no other explanation.

{numbered}

Output:"""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.models.generate_content(model=MODEL_ID, contents=prompt)
            text = resp.text
            # 空回應 = safety filter，不 retry，直接回傳 None list
            if not text or not text.strip():
                print(f"  [SKIP] Safety filter blocked this batch (empty response)")
                return [None] * len(batch_rows)
            lines = [l.strip() for l in text.strip().splitlines() if l.strip()]
            results = []
            for i in range(len(batch_rows)):
                matched = next((l for l in lines if l.startswith(f"{i+1}.")), None)
                results.append(matched.split(".", 1)[1].strip().strip('"').strip("'") if matched else None)
            # 部分缺失也接受，不 retry
            return results
        except Exception as e:
            # 503 / 429 等暫時性錯誤才 retry
            print(f"  Attempt {attempt}/{MAX_RETRIES} failed: {e}")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_SLEEP)
            else:
                print(f"  [SKIP] All retries exhausted, skipping batch")
                return [None] * len(batch_rows)

# 載入已完成的 checkpoint（斷點續跑）
done_ids = set()
rows = []
if CHECKPOINT_PATH.exists():
    with open(CHECKPOINT_PATH, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            rows.append(row)
            done_ids.add(row["id"])
    print(f"Resumed from checkpoint: {len(rows)} rows already done")

records = sampled.to_dict("records")

with open(CHECKPOINT_PATH, "a", encoding="utf-8") as ckpt:
    for lang in ["zh", "en"]:
        lang_records = [r for r in records if r["lang"] == lang and r["id"] not in done_ids]
        if not lang_records:
            print(f"{lang}: already done, skipping")
            continue
        for i in range(0, len(lang_records), BATCH_SIZE):
            batch = lang_records[i:i + BATCH_SIZE]
            plains = rewrite_batch(batch, lang)
            for r, plain in zip(batch, plains):
                row = {"orig_title": r["title"], "plain_title": plain,
                       "content": r.get("content", ""), "lang": r["lang"], "id": r["id"]}
                rows.append(row)
                ckpt.write(json.dumps(row, ensure_ascii=False) + "\n")
            ckpt.flush()
            print(f"{lang} {min(i + BATCH_SIZE, len(lang_records))}/{len(lang_records)}")
            time.sleep(SLEEP_BETWEEN_BATCHES)

import pandas as pd
rewritten = pd.DataFrame(rows)
print(f"\n成功改寫：{rewritten['plain_title'].notna().sum()} / {len(rewritten)}")
print(rewritten[["orig_title", "plain_title"]].head(10).to_string())

Resumed from checkpoint: 804 rows already done
zh 1/18
zh 2/18
zh 3/18
zh 4/18
zh 5/18
zh 6/18
zh 7/18
zh 8/18
zh 9/18
zh 10/18
zh 11/18
zh 12/18
zh 13/18
zh 14/18
zh 15/18
zh 16/18
zh 17/18
zh 18/18
en: already done, skipping

成功改寫：822 / 822
                        orig_title           plain_title
0              这种片能拍出来就赢了，何况还豆瓣9.4     紀錄片《守護者》介紹與案件背景概述
1                腰围不到1尺7？这是什么神仙身材！        鄭秀文分享日常運動與保養心得
2  口红界的大佬 ！全世界女人都想要的TomFord口红，4折抢！      Tom Ford唇彩限時促銷活動
3            做饭迷惑行为大赏：哈哈哈哈这是厨房杀手么？        網路蒐集各類居家烹飪失誤案例
4               黑话丨和男生做过最羞耻的事情是什么？        網友分享與異性互動的尷尬經驗
5             渣女最常祸害的3种男人，中招率达90%！      分析感情中常見的負面交往對象類型
6               黑话丨接吻时，男女都有什么生理反应？       網友分享接吻時的生理感受與反應
7      以“性”会友——知“性”更有“性”，欢迎加入地区交友群      介紹性學社群及其地區交友交流管道
8   那个和炮友在一起了的女人后来怎样了？我采访了5个有经历的女人     訪問五位曾與炮友交往者的經歷與心得
9        一天做3次爱会怎么样？上一个这样问的人，已经...  過度頻繁性行為對身體健康的影響與潛在危害


## Step 2b：補抽替換（safety filter 跳過 + 無效改寫）

將 checkpoint 中 `plain_title` 為 None，以及改寫結果無效（如「內容缺失」、「谢谢观看」等）的筆數，從未用過的 clickbait 中補抽替換，再重跑 Step 2。

In [3]:
import json, pandas as pd
from pathlib import Path

CHECKPOINT_PATH = Path("../dataset/processed/_rewrite_checkpoint.jsonl")

# 無效改寫的判斷條件
INVALID_PATTERNS = ["內容缺失", "无法提供", "谢谢观看", "暫無內容", "內容不足"]

def is_invalid(plain_title):
    if plain_title is None:
        return True
    for pat in INVALID_PATTERNS:
        if pat in plain_title:
            return True
    return False

# 讀 checkpoint，找出所有已用 id 和需要替換的 id
all_rows = []
with open(CHECKPOINT_PATH, encoding="utf-8") as f:
    for line in f:
        all_rows.append(json.loads(line))

all_used_ids = {r["id"] for r in all_rows}
failed_ids = {r["id"] for r in all_rows if is_invalid(r["plain_title"])}
print(f"需要替換：{len(failed_ids)} 筆")

if failed_ids:
    train = pd.read_csv("../dataset/processed/train.csv")
    click = train[train["label"] == 1]

    # 按語言統計需要替換的數量
    failed_by_lang = {}
    for r in all_rows:
        if r["id"] in failed_ids:
            failed_by_lang[r["lang"]] = failed_by_lang.get(r["lang"], 0) + 1

    # 從未用過的 pool 補抽
    new_records = []
    for lang, n in failed_by_lang.items():
        pool = click[(click["lang"] == lang) & (~click["id"].isin(all_used_ids))]
        replacements = pool.sample(n=n, random_state=99)
        new_records.extend(replacements.to_dict("records"))
        print(f"{lang}: 補抽 {n} 筆")

    # 把失敗的從 checkpoint 移除，更新 sampled 加入新 records
    cleaned = [r for r in all_rows if r["id"] not in failed_ids]
    with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
        for r in cleaned:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    # 更新 sampled 讓 Step 2 補跑新的
    new_df = pd.DataFrame(new_records)
    sampled = pd.concat([
        pd.DataFrame([r for r in cleaned]),
        new_df[["id", "lang", "title", "content", "label", "source"]]
    ]).reset_index(drop=True)

    print(f"Checkpoint 清理完成，請重新執行 Step 2 補跑 {len(new_records)} 筆")
else:
    print("無需替換")

需要替換：0 筆
無需替換


## Step 3：人工抽查

隨機抽取 50 筆輸出至 `_adversarial_tone_check.csv`，供人工確認改寫品質。

In [ ]:
sample_check = rewritten.dropna(subset=["plain_title"]).sample(n=50, random_state=1)
sample_check[["orig_title", "plain_title", "content"]].to_csv(
    "../dataset/processed/_adversarial_tone_check.csv", index=False)
print("抽查檔已輸出，請人工確認 plain_title 標 0 是否正確（資訊落差被填平）")

抽查檔已輸出，請人工確認 plain_title 標 0 是否正確（資訊落差被填平）


## Human Gate

請開啟 `dataset/processed/_adversarial_tone_check.csv` 人工確認改寫結果無誤後，再執行下一個 cell 繼續。

## Step 4：對抗資料補強（轉折詞反例）

針對 G7 觀察到的另一個盲點：**標題尾巴出現「竟然」、「但是」、「沒想到」等轉折詞時，
模型容易誤判為 clickbait（false positive）**。

策略：從 `train.csv` 抽 label=0 的真實非標題黨標題，請 Gemini 在句尾自然加上轉折詞收尾，
語意保持完整、不製造懸念缺口，產出 label=0 的反例，教模型「轉折詞 ≠ clickbait」。

- 中英文各 150 筆（合計 300 筆 label=0）
- id 後綴：`_tw_neg`（transition-word negative example）
- content 沿用原始來源新聞的 content

In [6]:
import json
import os
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from google import genai

load_dotenv(Path("../backend/.env"))
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

TW_MODEL_ID    = "gemini-3.1-flash-lite-preview"
TW_BATCH_SIZE  = 5
TW_SLEEP       = 6
TW_N_PER_LANG  = 150
TW_CHECKPOINT  = Path("../dataset/processed/_tw_neg_checkpoint.jsonl")

# 抽 label=0 的真實非標題黨標題作為來源
train = pd.read_csv("../dataset/processed/train.csv")
non_click = train[train["label"] == 0]
tw_zh = non_click[non_click["lang"] == "zh"].sample(n=TW_N_PER_LANG, random_state=7)
tw_en = non_click[non_click["lang"] == "en"].sample(n=TW_N_PER_LANG, random_state=7)
tw_sampled = pd.concat([tw_zh, tw_en]).reset_index(drop=True)
print(f"抽樣完成：zh={len(tw_zh)}, en={len(tw_en)}, 總計 {len(tw_sampled)}")


def add_transition_batch(batch_rows, lang):
    numbered = "\n".join(
        f"{i+1}. {r['title']}"
        for i, r in enumerate(batch_rows)
    )
    if lang == "zh":
        prompt = f"""你是新聞編輯。以下有 {len(batch_rows)} 個真實的非標題黨新聞標題（陳述事實型）。
請改寫每個標題，**讓「竟然 / 但是 / 沒想到 / 居然 / 然而 / 卻 / 雖然」這類轉折詞自然地出現在標題中**，
但改寫後的標題仍必須是**陳述完整事實**的非標題黨標題（不留懸念缺口、不誘導點擊）。

核心要求：轉折詞要扮演「事實對比」的功能（A 雖然 X 但實際是 Y、A 雖然 X 仍然 Y），
而不是當作句尾贅詞或感嘆語。

【好範例】（轉折詞嵌入事實對比，句意完整）：
- 原：北京2019年硕士研究生招生简章发布
  改：北京2019年硕士研究生招生简章发布，名额雖較去年减少但报考门槛保持不变
- 原：台股今日收盤上漲50點
  改：台股今日收盤雖一度回檔但終場仍上漲50點
- 原：新型冠状病毒疫情每日播报2020.02.03
  改：新型冠状病毒疫情每日播报2020.02.03，新增病例雖較昨日下降但累計人數仍超千人

【壞範例】（嚴禁這幾類！）：
- 「...，竟然如此」「...，事實確已如此」「...，居然」← 句尾贅詞，沒有真正轉折
- 「招生簡章公佈，內容竟然涵蓋這些要求」← 製造懸念缺口（什麼要求？）= 變成 clickbait
- 「演出現場居然呈現出這種效果」← 模糊代稱（什麼效果？）= 變成 clickbait

要求總結：
- 必須保留原標題的事實內容，不得加入原本沒有的事實
- 轉折詞必須連接兩個具體事實，不能是空泛的結語
- 不能出現「這種要求」「這種效果」「如此」「這樣」等模糊代稱結尾
- 改寫後仍是中性、可作為非標題黨的新聞標題
- 只輸出改寫後標題，每行一個，格式「編號. 改寫標題」，不要任何其他說明

{numbered}

輸出："""
    else:
        prompt = f"""You are a news editor. Below are {len(batch_rows)} real non-clickbait news headlines (factual statements).
Rewrite each so that a transition word (**"however", "but", "yet", "although", "despite", "still"**) appears naturally in the headline,
while the rewritten headline must remain a **complete factual statement** non-clickbait headline (no curiosity gap, no click bait).

Core requirement: the transition word must serve a **factual contrast** function (A although X, still Y / A despite X, remains Y),
NOT as a trailing filler word or exclamation.

【Good examples】(transition embedded in factual contrast, complete meaning):
- Original: Apple released the iPhone 15 with a starting price of $799
  Rewrite: Apple released the iPhone 15 at a $799 starting price, although the Pro model price increased by $100
- Original: Tesla Q3 earnings beat analyst expectations
  Rewrite: Tesla Q3 earnings beat analyst expectations despite a 6% drop in vehicle deliveries
- Original: US inflation rate fell to 3.2% in October
  Rewrite: US inflation rate fell to 3.2% in October although core CPI remained above target

【Bad examples】(STRICTLY FORBIDDEN!):
- "..., however that was the case" / "..., surprisingly so" / "..., yet it remained true" ← trailing filler, no real contrast
- "Apple released a new phone, however the response was unexpected" ← curiosity gap (what response?) = becomes clickbait
- "The conference yielded surprising outcomes, however" ← vague reference (what outcomes?) = becomes clickbait

Final requirements:
- Preserve the original factual content; do not add new facts not in the original
- The transition word must connect two concrete facts, not be an empty closing phrase
- Do NOT end with vague references like "such results", "this outcome", "as expected", "in this way"
- Rewritten headline must remain a neutral non-clickbait news headline
- Output only rewritten headlines, one per line, format "N. rewritten headline", no other text

{numbered}

Output:"""

    for attempt in range(1, 4):
        try:
            resp = client.models.generate_content(model=TW_MODEL_ID, contents=prompt)
            text = resp.text
            if not text or not text.strip():
                print("  [SKIP] empty response (safety filter)")
                return [None] * len(batch_rows)
            lines = [l.strip() for l in text.strip().splitlines() if l.strip()]
            results = []
            for i in range(len(batch_rows)):
                matched = next((l for l in lines if l.startswith(f"{i+1}.")), None)
                results.append(
                    matched.split(".", 1)[1].strip().strip('"').strip("'")
                    if matched else None
                )
            return results
        except Exception as e:
            print(f"  attempt {attempt}/3 failed: {e}")
            if attempt < 3:
                time.sleep(15)
            else:
                return [None] * len(batch_rows)


# 載入已完成 checkpoint（斷點續跑）；若要重跑請手動刪 _tw_neg_checkpoint.jsonl
tw_done_ids = set()
tw_rows = []
if TW_CHECKPOINT.exists():
    with open(TW_CHECKPOINT, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            tw_rows.append(row)
            tw_done_ids.add(row["id"])
    print(f"從 checkpoint 續跑：已完成 {len(tw_rows)} 筆")

records = tw_sampled.to_dict("records")

with open(TW_CHECKPOINT, "a", encoding="utf-8") as ckpt:
    for lang in ["zh", "en"]:
        lang_records = [r for r in records if r["lang"] == lang and r["id"] not in tw_done_ids]
        if not lang_records:
            print(f"{lang}: 已完成，跳過")
            continue
        for i in range(0, len(lang_records), TW_BATCH_SIZE):
            batch = lang_records[i:i + TW_BATCH_SIZE]
            rewrites = add_transition_batch(batch, lang)
            for r, rewritten_title in zip(batch, rewrites):
                row = {
                    "id":         r["id"],
                    "lang":       r["lang"],
                    "orig_title": r["title"],
                    "tw_title":   rewritten_title,
                    "content":    r.get("content", ""),
                }
                tw_rows.append(row)
                ckpt.write(json.dumps(row, ensure_ascii=False) + "\n")
            ckpt.flush()
            print(f"{lang} {min(i + TW_BATCH_SIZE, len(lang_records))}/{len(lang_records)}")
            time.sleep(TW_SLEEP)

tw_rewritten = pd.DataFrame(tw_rows)
print(f"\n成功改寫：{tw_rewritten['tw_title'].notna().sum()} / {len(tw_rewritten)}")
print(tw_rewritten[["orig_title", "tw_title"]].head(10).to_string())

抽樣完成：zh=150, en=150, 總計 300
zh 5/150
zh 10/150
zh 15/150
zh 20/150
zh 25/150
zh 30/150
zh 35/150
zh 40/150
zh 45/150
zh 50/150
zh 55/150
zh 60/150
zh 65/150
zh 70/150
zh 75/150
zh 80/150
zh 85/150
zh 90/150
zh 95/150
zh 100/150
zh 105/150
zh 110/150
zh 115/150
zh 120/150
zh 125/150
zh 130/150
zh 135/150
zh 140/150
zh 145/150
zh 150/150
en 5/150
en 10/150
en 15/150
en 20/150
en 25/150
en 30/150
en 35/150
en 40/150
en 45/150
en 50/150
en 55/150
en 60/150
en 65/150
en 70/150
en 75/150
en 80/150
en 85/150
en 90/150
en 95/150
en 100/150
en 105/150
en 110/150
en 115/150
en 120/150
en 125/150
en 130/150
en 135/150
en 140/150
en 145/150
en 150/150

成功改寫：300 / 300
                                      orig_title                                         tw_title
0                 首次披露：“7·5”事件、北京暴恐案及昆明暴恐案部分原始视频     “7·5”事件、北京及昆明暴恐案原始视频首次披露，虽画面内容极具冲击力但均为重要调查证据
1                                理以权度•得其所益|食品安全篇            食品安全监管行动持续推进，虽企业理应从权责中获益但仍需通过严格审查方能达标
2                     北京丰台花乡升级为疫情高风险地区！新增7

## Step 5：組成對抗資料並輸出

將以下三類資料合併輸出至 `dataset/processed/adversarial_tone.csv`：

1. **Tone 對抗（原版誇張）**：`{原id}_adv_orig`，label=1
2. **Tone 對抗（平實改寫）**：`{原id}_adv_plain`，label=0
3. **轉折詞反例**：`{原id}_tw_neg`，label=0（教模型「結尾轉折詞 ≠ clickbait」）

In [ ]:
# --- 1. Tone 對抗：原版誇張 (label=1) + 平實改寫 (label=0) ---
ok = rewritten.dropna(subset=["plain_title"])
ok = ok[ok["plain_title"].str.len() > 0]

orig = pd.DataFrame({
    "id":      ok["id"].astype(str) + "_adv_orig",
    "lang":    ok["lang"],
    "title":   ok["orig_title"],
    "content": ok["content"],
    "label":   1,
    "source":  "adversarial_tone",
})
plain = pd.DataFrame({
    "id":      ok["id"].astype(str) + "_adv_plain",
    "lang":    ok["lang"],
    "title":   ok["plain_title"],
    "content": ok["content"],
    "label":   0,
    "source":  "adversarial_tone",
})

# --- 2. 轉折詞反例 (label=0) ---
tw_ok = tw_rewritten.dropna(subset=["tw_title"])
tw_ok = tw_ok[tw_ok["tw_title"].str.len() > 0]

tw_neg = pd.DataFrame({
    "id":      tw_ok["id"].astype(str) + "_tw_neg",
    "lang":    tw_ok["lang"],
    "title":   tw_ok["tw_title"],
    "content": tw_ok["content"],
    "label":   0,
    "source":  "adversarial_tone",
})

# --- 合併 ---
adv = pd.concat([orig, plain, tw_neg]).reset_index(drop=True)
adv.to_csv("../dataset/processed/adversarial_tone.csv", index=False)

assert set(adv["label"].unique()) == {0, 1}, "label 必須只有 0/1"
assert list(adv.columns) == ["id", "lang", "title", "content", "label", "source"]
assert adv["lang"].isin(["zh", "en"]).all()
assert adv["id"].is_unique, "id 必須唯一"

print(f"對抗資料總筆數：{len(adv)}")
print(f"  - tone orig (label=1)  : {len(orig)}")
print(f"  - tone plain (label=0) : {len(plain)}")
print(f"  - tw neg (label=0)     : {len(tw_neg)}")
print(f"  - 語言分布：\n{adv['lang'].value_counts()}")
print(f"  - label 分布：\n{adv['label'].value_counts()}")

對抗資料總筆數：1944
  - tone orig (label=1)  : 822
  - tone plain (label=0) : 822
  - tw neg (label=0)     : 300
  - 語言分布：
lang
zh    994
en    950
Name: count, dtype: int64
  - label 分布：
label
0    1122
1     822
Name: count, dtype: int64
